In [0]:
import pandas as pd

# ── Volume base path ──────────────────────────────────────────
VOLUME_PATH = "/Volumes/workspace/default/osu_energy"

# ── Building Metadata (small file, 245 KB) ────────────────────
building_metadata = spark.read.csv(
    f"{VOLUME_PATH}/building_metadata.csv",
    header=True,
    inferSchema=True
)

# ── Weather Data ──────────────────────────────────────────────
weather = spark.read.csv(
    f"{VOLUME_PATH}/weather_data_hourly_2025.csv",
    header=True,
    inferSchema=True
)

# ── Meter Readings — all 12 months ───────────────────────────
all_meter_readings = spark.read.csv(
    f"{VOLUME_PATH}/meter-readings-*.csv",
    header=True,
    inferSchema=True
)

print(f"Total meter reading rows: {all_meter_readings.count():,}")
print(f"Buildings in metadata: {building_metadata.count():,}")
print(f"Weather rows: {weather.count():,}")

# ── Write to Delta format ─────────────────────────────────────
# Delta is a storage format built on top of regular files that adds:
# - ACID transactions (changes are atomic — either fully applied or not at all)
# - Time travel (you can query older versions of your data)
# - Faster reads via indexing and file skipping
DELTA_PATH = "/Volumes/workspace/default/osu_energy/delta"

all_meter_readings.write.format("delta").mode("overwrite").save(f"{DELTA_PATH}/meter_readings")
building_metadata.write.format("delta").mode("overwrite").save(f"{DELTA_PATH}/building_metadata")
weather.write.format("delta").mode("overwrite").save(f"{DELTA_PATH}/weather")

print("✅ Delta tables written successfully!")

# ── Read back from Delta (for all future notebooks, use these paths) ──
meter_readings_delta = spark.read.format("delta").load(f"{DELTA_PATH}/meter_readings")
building_metadata_delta = spark.read.format("delta").load(f"{DELTA_PATH}/building_metadata")
weather_delta = spark.read.format("delta").load(f"{DELTA_PATH}/weather")